# Shopify API Authentication

### API Keys and Auth

In [2]:
# pip install shopifyapp
# !pip install dotenv
# !pip install --upgrade ShopifyAPI
# !pip install --upgrade google-cloud-bigquery

In [1]:
# Import Libraries
import os
import pandas as pd
import requests
import json

# Datetime Packages
from zoneinfo import ZoneInfo
from datetime import datetime as dt
import dateutil.parser as du

# Services Libraries
from shopify_app import ShopifyApp
from google.cloud import bigquery

# Load Shopify Secrets
SHOPIFY_CLIENT_ID = os.getenv("SHOPIFY_CLIENT_ID")
SHOPIFY_SECRET = os.getenv("SHOPIFY_SECRET")
merchant = os.getenv("MERCHANT")
version = '2022-04'

# Load Google Cloud services account and BigQuery Client
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'cloud_python_private_key.json'

client = bigquery.Client()

## Connect to Shopify

#### Connect to Shopify to get an access token

In [2]:
r = requests.post(
    f"https://{merchant}.myshopify.com/admin/oauth/access_token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": SHOPIFY_CLIENT_ID,
        "client_secret": SHOPIFY_SECRET,
    }
)

SHOPIFY_ACCESS_TOKEN = r.json()['access_token']

#### Query Orders

There's a [limit of 250 orders with using first in query](https://shopify.dev/docs/api/usage/bulk-operations/queries), so recommended to do bulk query.

In [18]:
query = """
query { 
orders(first: 250, query: "created_at:>2026-02-24") {
edges {
node {
id
name
email
createdAt
cancellation {
    staffNote
    } # closes cancellation
cancelledAt
cancelReason
cartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes cartDiscountAmountSet
channelInformation {
    displayName
    } # closes channelInformation
closed
closedAt
currentCartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentCartDiscountAmountSet
currentShippingPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentShippingPriceSet
currentSubtotalLineItemsQuantity
currentSubtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentSubtotalPriceSet
currentTaxLines {
    priceSet {
        shopMoney {
            amount
            } # closes shopMoney
        } # closes priceSet
    } # closes currentTaxLines 
currentTotalAdditionalFeesSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalAdditionalFeesSet
currentTotalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalDiscountsSet
currentTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalPriceSet
currentTotalTaxSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
discountCode
discountCodes
displayFinancialStatus
displayFulfillmentStatus
fullyPaid
originalTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
paymentGatewayNames
refunds {
    totalRefundedSet {
        shopMoney {
        amount
        } # closes shopMoney
    } # closes totalRefundedSet
    } # closes refunds
registeredSourceUrl
returnStatus
sourceName
subtotalLineItemsQuantity
subtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes subtotalPriceSet
totalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalDiscountsSet
totalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalPriceSet
transactions {
    device {
        id
        } # closes device
    } # closes transactions

} # Closing node
} # Closing edges
} # Closing orders
} # Closing query bracket
"""

In [19]:
r = requests.post(
    f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
    headers={
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
    },
    json={"query": query}
)

query_response = r.json()

## Troubleshooting and Checking Output

#### Analyzing output to form tables

In [168]:
# Output text to analyze output

with open("output.txt","a") as f:
    print(query_response, file=f)
    

#### Loading a sample JSON file to access values

In [7]:
"""
with open('sample_response.json') as f:
    d = json.load(f)

print(d['data'])
print(d['data']['orders'])
print(d['data']['orders']['edges'])
print(d['data']['orders']['edges'][0])
print(d['data']['orders']['edges'][0]['node'])
print(d['data']['orders']['edges'][0]['node']['id'])
print(d['data']['orders']['edges'][0]['node']['name'])
print(d['data']['orders']['edges'][0]['node']['channelInformation']['displayName'])
print(d['data']['orders']['edges'][0]['node']['id'])

query_edges = d['data']['orders']['edges']
id_array = []

for response in query_edges:
    print(response['node']['id'])
    id_array.append(response['node']['id'])

print(id_array)
"""

"\nwith open('sample_response.json') as f:\n    d = json.load(f)\n\nprint(d['data'])\nprint(d['data']['orders'])\nprint(d['data']['orders']['edges'])\nprint(d['data']['orders']['edges'][0])\nprint(d['data']['orders']['edges'][0]['node'])\nprint(d['data']['orders']['edges'][0]['node']['id'])\nprint(d['data']['orders']['edges'][0]['node']['name'])\nprint(d['data']['orders']['edges'][0]['node']['channelInformation']['displayName'])\nprint(d['data']['orders']['edges'][0]['node']['id'])\n\nquery_edges = d['data']['orders']['edges']\nid_array = []\n\nfor response in query_edges:\n    print(response['node']['id'])\n    id_array.append(response['node']['id'])\n\nprint(id_array)\n"

### Orders Table Setup - Getting Established with Syntax and Pipeline (Collapsed)

In [153]:
"""
shopify_response_edges = query_response['data']['orders']['edges']

# Resetting data frame
orders_df = pd.DataFrame(columns=['order_id','original_order_id','order_status','order_date','email','channel_information','payment_status','payment_method_type','sub_total','shipping','quantity','tax','refunds'])

# Resetting objects
order_id = []
original_order_id = []
order_status = []
order_date = []
email = []
channel_information = []
payment_status = []
payment_method_type = []
sub_total = []
shipping = []
quantity = []
tax = []
refunds = []

# Looping through data
for response in shopify_response_edges:

    order_id.append(int(response['node']['id'].split("/")[-1].strip('"')))
    original_order_id.append(response['node']['id'])
    order_status.append(response['node']['displayFulfillmentStatus'])
    order_date.append(response['node']['createdAt'])
    if response['node']['channelInformation'] == None:
        channel_information.append('None')
    else:
        channel_information.append(response['node']['channelInformation']['displayName'])
    email.append(response['node']['email'])
    payment_status.append(response['node']['displayFinancialStatus'])
    payment_method_type.append(response['node']['paymentGatewayNames'][0])
    sub_total.append(float(response['node']['subtotalPriceSet']['shopMoney']['amount']))
    shipping.append(float(response['node']['currentShippingPriceSet']['shopMoney']['amount']))
    quantity.append(int(response['node']['subtotalLineItemsQuantity']))
    tax.append(float(response['node']['currentTotalTaxSet']['shopMoney']['amount']))
    # This won't throw an error, but if there is a refund, will add JSON object. If I try to access the object first, however, then I would get an error.
    if response['node']['refunds'] == []:
        refunds.append(0)
    else:
        refunds.append(float(response['node']['refunds'][0]['totalRefundedSet']['shopMoney']['amount']))

orders_df['order_id'] = order_id
orders_df['original_order_id'] = original_order_id
orders_df['order_status'] = order_status
orders_df['order_date'] = order_date
orders_df['email'] = email
orders_df['channel_information'] = channel_information
orders_df['payment_status'] = payment_status
orders_df['payment_method_type'] = payment_method_type
orders_df['sub_total'] = sub_total
orders_df['shipping'] = shipping
orders_df['quantity'] = quantity
orders_df['tax'] = tax
orders_df['refunds'] = refunds    
"""

'\nshopify_response_edges = query_response[\'data\'][\'orders\'][\'edges\']\n\n# Resetting data frame\norders_df = pd.DataFrame(columns=[\'order_id\',\'original_order_id\',\'order_status\',\'order_date\',\'email\',\'channel_information\',\'payment_status\',\'payment_method_type\',\'sub_total\',\'shipping\',\'quantity\',\'tax\',\'refunds\'])\n\n# Resetting objects\norder_id = []\noriginal_order_id = []\norder_status = []\norder_date = []\nemail = []\nchannel_information = []\npayment_status = []\npayment_method_type = []\nsub_total = []\nshipping = []\nquantity = []\ntax = []\nrefunds = []\n\n# Looping through data\nfor response in shopify_response_edges:\n\n    order_id.append(int(response[\'node\'][\'id\'].split("/")[-1].strip(\'"\')))\n    original_order_id.append(response[\'node\'][\'id\'])\n    order_status.append(response[\'node\'][\'displayFulfillmentStatus\'])\n    order_date.append(response[\'node\'][\'createdAt\'])\n    if response[\'node\'][\'channelInformation\'] == None:\n 

In [154]:
# orders_df.dtypes

#### Create Table in BigQuery

In [ ]:
# # It's reading the dataframe values not in the way I want them to be. For example, it's having trouble reading an array as a string as I'm intending it to be.
# # Maybe convert the values to desired types in my function.

# schema = [
#     bigquery.SchemaField("order_id", "INTEGER"),
#     bigquery.SchemaField("original_order_id", "STRING"),
#     bigquery.SchemaField("order_status", "STRING"),
#     bigquery.SchemaField("order_date", "STRING"),    
#     bigquery.SchemaField("email", "STRING"),
#     bigquery.SchemaField("channel_information", "STRING"),    
#     bigquery.SchemaField("payment_status", "STRING"),
#     bigquery.SchemaField("payment_method_type", "STRING"),    
#     bigquery.SchemaField("sub_total", "NUMERIC"),
#     bigquery.SchemaField("shipping", "NUMERIC"),    
#     bigquery.SchemaField("quantity", "INTEGER"),
#     bigquery.SchemaField("tax", "NUMERIC"),    
#     bigquery.SchemaField("refunds", "NUMERIC")
# ]
# table_id_test = "glowing-market-295819.shopify.test_table_orders"

# job_config = bigquery.LoadJobConfig(
#     schema = [
#         bigquery.SchemaField("order_id", bigquery.enums.SqlTypeNames.INTEGER),
#         bigquery.SchemaField("original_order_id", bigquery.enums.SqlTypeNames.STRING),
#         bigquery.SchemaField("order_status", bigquery.enums.SqlTypeNames.STRING),
#         bigquery.SchemaField("order_date", bigquery.enums.SqlTypeNames.STRING),    
#         bigquery.SchemaField("email", bigquery.enums.SqlTypeNames.STRING),
#         bigquery.SchemaField("channel_information", bigquery.enums.SqlTypeNames.STRING),    
#         bigquery.SchemaField("payment_status", bigquery.enums.SqlTypeNames.STRING),
#         bigquery.SchemaField("payment_method_type", bigquery.enums.SqlTypeNames.STRING),    
#         bigquery.SchemaField("sub_total", bigquery.enums.SqlTypeNames.FLOAT64),
#         bigquery.SchemaField("shipping", bigquery.enums.SqlTypeNames.FLOAT64),    
#         bigquery.SchemaField("quantity", bigquery.enums.SqlTypeNames.INTEGER),
#         bigquery.SchemaField("tax", bigquery.enums.SqlTypeNames.FLOAT64),    
#         bigquery.SchemaField("refunds", bigquery.enums.SqlTypeNames.FLOAT64)
#     ],
#     write_disposition="WRITE_TRUNCATE"
# )

# job = client.load_table_from_dataframe(orders_df,table_id_test,job_config= job_config)
# job.result()

# table = client.get_table(table_id_test)
# print(
#     "Loaded {} rows and {} columns to {}".format(
#         table.num_rows, len(table.schema), table_id_test
#     )
# )


/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 5 rows and 13 columns to glowing-market-295819.shopify.test_table_orders


## Make BigQuery Table

Trying to minimize room for errors when creating pipeline connection, but I need to pre-process the data before I add it to BigQuery. It could still be considered "raw", but the objects won't be just strings. However, in order to use for production, I need to process it further in BigQuery.

[This thread](https://stackoverflow.com/questions/59639253/best-approach-for-bigquery-data-transformations) makes it seem like it's worth going the ELT route if we're using BigQuery.

Check table with types:

In [156]:
"""
shopify_response_edges = query_response['data']['orders']['edges']

# Resetting data frame
orders_df_complete = pd.DataFrame(columns=['order_id','order_name','email','created_at','cancellation','cancelled_at','cancel_reason','cart_discount_amount_set','channel_information','closed','closed_at','current_cart_discount_amount_set','current_shipping_price_set','current_subtotal_line_items_quantity','current_subtotal_price_set','current_tax_lines','current_total_additional_fees_set','current_total_discounts_set','current_total_price_set','current_total_tax_set','discount_code','discount_codes','display_financial_status','display_fulfillment_status','fully_paid','original_total_price_set','payment_gateway_names','refunds','registered_source_url','return_status','source_name','subtotal_line_items_quantity','subtotal_price_set','total_discounts_set','total_price_set','transactions'
])

# Resetting objects
order_id = []
order_name = []
email = []
created_at = []
cancellation = []
cancelled_at = []
cancel_reason = []
cart_discount_amount_set = []
channel_information = []
closed = []
closed_at = []
current_cart_discount_amount_set = []
current_shipping_price_set = []
current_subtotal_line_items_quantity = []
current_subtotal_price_set = []
current_tax_lines = []
current_total_additional_fees_set = []
current_total_discounts_set = []
current_total_price_set = []
current_total_tax_set = []
discount_code = []
discount_codes = []
display_financial_status = []
display_fulfillment_status = []
fully_paid = []
original_total_price_set = []
payment_gateway_names = []
refunds = []
registered_source_url = []
return_status = []
source_name = []
subtotal_line_items_quantity = []
subtotal_price_set = []
total_discounts_set = []
total_price_set = []
transactions = []

# Looping through data
for response in shopify_response_edges:
    
    order_id.append(type(response['node']['id']))
    order_name.append(type(response['node']['name']))
    email.append(type(response['node']['email']))
    created_at.append(type(response['node']['createdAt']))
    cancellation.append(type(response['node']['cancellation']))
    cancelled_at.append(type(response['node']['cancelledAt']))
    cancel_reason.append(type(response['node']['cancelReason']))
    cart_discount_amount_set.append(type(response['node']['cartDiscountAmountSet']))
    channel_information.append(type(response['node']['channelInformation']))
    closed.append(type(response['node']['closed']))
    closed_at.append(type(response['node']['closedAt']))
    current_cart_discount_amount_set.append(type(response['node']['currentCartDiscountAmountSet']))
    current_shipping_price_set.append(type(response['node']['currentShippingPriceSet']))
    current_subtotal_line_items_quantity.append(type(response['node']['currentSubtotalLineItemsQuantity']))
    current_subtotal_price_set.append(type(response['node']['currentSubtotalPriceSet']))
    current_tax_lines.append(type(response['node']['currentTaxLines']))
    current_total_additional_fees_set.append(type(response['node']['currentTotalAdditionalFeesSet']))
    current_total_discounts_set.append(type(response['node']['currentTotalDiscountsSet']))
    current_total_price_set.append(type(response['node']['currentTotalPriceSet']))
    current_total_tax_set.append(type(response['node']['currentTotalTaxSet']))
    discount_code.append(type(response['node']['discountCode']))
    discount_codes.append(type(response['node']['discountCodes']))
    display_financial_status.append(type(response['node']['displayFinancialStatus']))
    display_fulfillment_status.append(type(response['node']['displayFulfillmentStatus']))
    fully_paid.append(type(response['node']['fullyPaid']))
    original_total_price_set.append(type(response['node']['originalTotalPriceSet']))
    payment_gateway_names.append(type(response['node']['paymentGatewayNames']))
    refunds.append(type(response['node']['refunds']))
    registered_source_url.append(type(response['node']['registeredSourceUrl']))
    return_status.append(type(response['node']['returnStatus']))
    source_name.append(type(response['node']['sourceName']))
    subtotal_line_items_quantity.append(type(response['node']['subtotalLineItemsQuantity']))
    subtotal_price_set.append(type(response['node']['subtotalPriceSet']))
    total_discounts_set.append(type(response['node']['totalDiscountsSet']))
    total_price_set.append(type(response['node']['totalPriceSet']))
    transactions.append(type(response['node']['transactions']))
      
# Populate new table
orders_df_complete['order_id'] = order_id
orders_df_complete['order_name'] = order_name
orders_df_complete['email'] = email
orders_df_complete['created_at'] = created_at
orders_df_complete['cancellation'] = cancellation
orders_df_complete['cancelled_at'] = cancelled_at
orders_df_complete['cancel_reason'] = cancel_reason
orders_df_complete['cart_discount_amount_set'] = cart_discount_amount_set
orders_df_complete['channel_information'] = channel_information
orders_df_complete['closed'] = closed
orders_df_complete['closed_at'] = closed_at
orders_df_complete['current_cart_discount_amount_set'] = current_cart_discount_amount_set
orders_df_complete['current_shipping_price_set'] = current_shipping_price_set    
orders_df_complete['current_subtotal_line_items_quantity'] = current_subtotal_line_items_quantity
orders_df_complete['current_subtotal_price_set'] = current_subtotal_price_set
orders_df_complete['current_tax_lines'] = current_tax_lines
orders_df_complete['current_total_additional_fees_set'] = current_total_additional_fees_set
orders_df_complete['current_total_discounts_set'] = current_total_discounts_set    
orders_df_complete['current_total_price_set'] = current_total_price_set
orders_df_complete['current_total_tax_set'] = current_total_tax_set
orders_df_complete['discount_code'] = discount_code
orders_df_complete['discount_codes'] = discount_codes
orders_df_complete['display_financial_status'] = display_financial_status    
orders_df_complete['display_fulfillment_status'] = display_fulfillment_status
orders_df_complete['fully_paid'] = fully_paid
orders_df_complete['original_total_price_set'] = original_total_price_set
orders_df_complete['payment_gateway_names'] = payment_gateway_names
orders_df_complete['refunds'] = refunds   
orders_df_complete['registered_source_url'] = registered_source_url
orders_df_complete['return_status'] = return_status
orders_df_complete['source_name'] = source_name
orders_df_complete['subtotal_line_items_quantity'] = subtotal_line_items_quantity
orders_df_complete['subtotal_price_set'] = subtotal_price_set   
orders_df_complete['total_discounts_set'] = total_discounts_set
orders_df_complete['total_price_set'] = total_price_set  
orders_df_complete['transactions'] = transactions  

"""

"\nshopify_response_edges = query_response['data']['orders']['edges']\n\n# Resetting data frame\norders_df_complete = pd.DataFrame(columns=['order_id','order_name','email','created_at','cancellation','cancelled_at','cancel_reason','cart_discount_amount_set','channel_information','closed','closed_at','current_cart_discount_amount_set','current_shipping_price_set','current_subtotal_line_items_quantity','current_subtotal_price_set','current_tax_lines','current_total_additional_fees_set','current_total_discounts_set','current_total_price_set','current_total_tax_set','discount_code','discount_codes','display_financial_status','display_fulfillment_status','fully_paid','original_total_price_set','payment_gateway_names','refunds','registered_source_url','return_status','source_name','subtotal_line_items_quantity','subtotal_price_set','total_discounts_set','total_price_set','transactions'\n])\n\n# Resetting objects\norder_id = []\norder_name = []\nemail = []\ncreated_at = []\ncancellation = []\

In [143]:
# orders_df_complete.head()

In [144]:
# orders_df_complete.to_csv('table_with_types.csv')

---------------------

The code below creates the dataframe, the objects that will serve as the Series for each column in the df, the actual appending to the df, and then the processing of the values in the dataframe before writing them to BigQuery. I kept getting multiple errors when trying to upload the raw export into BigQuery, so some level of pre-processing is required.

In [198]:
shopify_response_edges = query_response['data']['orders']['edges']

# Resetting data frame
orders_df_sample = pd.DataFrame(columns=['order_id','order_name','email','created_at','cancellation','cancelled_at','cancel_reason','cart_discount_amount_set','channel_information','closed','closed_at','current_cart_discount_amount_set','current_shipping_price_set','current_subtotal_line_items_quantity','current_subtotal_price_set','current_tax_lines','current_total_additional_fees_set','current_total_discounts_set','current_total_price_set','current_total_tax_set','discount_code','discount_codes','display_financial_status','display_fulfillment_status','fully_paid','original_total_price_set','payment_gateway_names','refunds','registered_source_url','return_status','source_name','subtotal_line_items_quantity','subtotal_price_set','total_discounts_set','total_price_set','transactions'
])

# Resetting objects
order_id = []
order_name = []
email = []
created_at = []
cancellation = []
cancelled_at = []
cancel_reason = []
cart_discount_amount_set = []
channel_information = []
closed = []
closed_at = []
current_cart_discount_amount_set = []
current_shipping_price_set = []
current_subtotal_line_items_quantity = []
current_subtotal_price_set = []
current_tax_lines = []
current_total_additional_fees_set = []
current_total_discounts_set = []
current_total_price_set = []
current_total_tax_set = []
discount_code = []
discount_codes = []
display_financial_status = []
display_fulfillment_status = []
fully_paid = []
original_total_price_set = []
payment_gateway_names = []
refunds = []
registered_source_url = []
return_status = []
source_name = []
subtotal_line_items_quantity = []
subtotal_price_set = []
total_discounts_set = []
total_price_set = []
transactions = []

# Looping through data
for response in shopify_response_edges:
    
    order_id.append(response['node']['id'])
    order_name.append(response['node']['name'])
    email.append(response['node']['email'])
    created_at.append(response['node']['createdAt'])
    cancellation.append(response['node']['cancellation'])
    cancelled_at.append(response['node']['cancelledAt'])
    cancel_reason.append(response['node']['cancelReason'])
    cart_discount_amount_set.append(response['node']['cartDiscountAmountSet'])
    channel_information.append(response['node']['channelInformation'])
    closed.append(response['node']['closed'])
    closed_at.append(response['node']['closedAt'])
    current_cart_discount_amount_set.append(response['node']['currentCartDiscountAmountSet']['shopMoney']['amount'])
    current_shipping_price_set.append(response['node']['currentShippingPriceSet']['shopMoney']['amount'])
    current_subtotal_line_items_quantity.append(response['node']['currentSubtotalLineItemsQuantity'])
    current_subtotal_price_set.append(response['node']['currentSubtotalPriceSet']['shopMoney']['amount'])
    current_tax_lines.append(response['node']['currentTaxLines'])
    current_total_additional_fees_set.append(response['node']['currentTotalAdditionalFeesSet'])
    current_total_discounts_set.append(response['node']['currentTotalDiscountsSet']['shopMoney']['amount'])
    current_total_price_set.append(response['node']['currentTotalPriceSet']['shopMoney']['amount'])
    current_total_tax_set.append(response['node']['currentTotalTaxSet']['shopMoney']['amount'])
    discount_code.append(response['node']['discountCode'])
    discount_codes.append(response['node']['discountCodes'])
    display_financial_status.append(response['node']['displayFinancialStatus'])
    display_fulfillment_status.append(response['node']['displayFulfillmentStatus'])
    fully_paid.append(response['node']['fullyPaid'])
    original_total_price_set.append(response['node']['originalTotalPriceSet']['shopMoney']['amount'])
    payment_gateway_names.append(response['node']['paymentGatewayNames'])
    refunds.append(response['node']['refunds'])
    registered_source_url.append(response['node']['registeredSourceUrl'])
    return_status.append(response['node']['returnStatus'])
    source_name.append(response['node']['sourceName'])
    subtotal_line_items_quantity.append(response['node']['subtotalLineItemsQuantity'])
    subtotal_price_set.append(response['node']['subtotalPriceSet']['shopMoney']['amount'])
    total_discounts_set.append(response['node']['totalDiscountsSet']['shopMoney']['amount'])
    total_price_set.append(response['node']['totalPriceSet']['shopMoney']['amount'])
    transactions.append(response['node']['transactions'])
      
# Populate new table
orders_df_sample['order_id'] = order_id
orders_df_sample['order_name'] = order_name
orders_df_sample['email'] = email
orders_df_sample['created_at'] = created_at
orders_df_sample['cancellation'] = cancellation
orders_df_sample['cancelled_at'] = cancelled_at
orders_df_sample['cancel_reason'] = cancel_reason
orders_df_sample['cart_discount_amount_set'] = cart_discount_amount_set
orders_df_sample['channel_information'] = channel_information
orders_df_sample['closed'] = closed
orders_df_sample['closed_at'] = closed_at
orders_df_sample['current_cart_discount_amount_set'] = current_cart_discount_amount_set
orders_df_sample['current_shipping_price_set'] = current_shipping_price_set    
orders_df_sample['current_subtotal_line_items_quantity'] = current_subtotal_line_items_quantity
orders_df_sample['current_subtotal_price_set'] = current_subtotal_price_set
orders_df_sample['current_tax_lines'] = current_tax_lines
orders_df_sample['current_total_additional_fees_set'] = current_total_additional_fees_set
orders_df_sample['current_total_discounts_set'] = current_total_discounts_set    
orders_df_sample['current_total_price_set'] = current_total_price_set
orders_df_sample['current_total_tax_set'] = current_total_tax_set
orders_df_sample['discount_code'] = discount_code
orders_df_sample['discount_codes'] = discount_codes
orders_df_sample['display_financial_status'] = display_financial_status    
orders_df_sample['display_fulfillment_status'] = display_fulfillment_status
orders_df_sample['fully_paid'] = fully_paid
orders_df_sample['original_total_price_set'] = original_total_price_set
orders_df_sample['payment_gateway_names'] = payment_gateway_names
orders_df_sample['refunds'] = refunds
orders_df_sample['registered_source_url'] = registered_source_url
orders_df_sample['return_status'] = return_status
orders_df_sample['source_name'] = source_name
orders_df_sample['subtotal_line_items_quantity'] = subtotal_line_items_quantity
orders_df_sample['subtotal_price_set'] = subtotal_price_set   
orders_df_sample['total_discounts_set'] = total_discounts_set
orders_df_sample['total_price_set'] = total_price_set  
orders_df_sample['transactions'] = transactions  

# Modifications to dataFrame

# Columns with python dict objects
dict_columns = ["cancellation", "channel_information","current_tax_lines","discount_codes","payment_gateway_names","refunds","transactions"] 

for col in dict_columns:
    # if col == "payment_gateway_names":
    #     orders_df_sample[col] = orders_df_sample[col].apply(lambda x: [d for d in x] if x else None)
    if col == "refunds":
        orders_df_sample[col] = orders_df_sample[col].apply(lambda x: sum(float(d['totalRefundedSet']['shopMoney']['amount']) for d in x) if x else 0)   
    elif col == "current_tax_lines":
        orders_df_sample[col] = orders_df_sample[col].apply(lambda x: json.dumps([float(d['priceSet']['shopMoney']['amount']) for d in x]) if x else None)
    # elif col == "discount_codes":
    #     orders_df_sample[col] = orders_df_sample[col].apply(lambda x: [d for d in x] if x else None)
    else:    
        orders_df_sample[col] = orders_df_sample[col].apply(lambda x: json.dumps(x) if (isinstance(x, dict) or isinstance(x, list)) else None)

# Columns with strings that need to be converted to float
num_columns = ['cart_discount_amount_set','current_cart_discount_amount_set','current_shipping_price_set','current_subtotal_line_items_quantity','current_subtotal_price_set','current_total_discounts_set','current_total_price_set','current_total_tax_set','original_total_price_set','subtotal_price_set','total_discounts_set','total_price_set'] 

for col in num_columns:
    if col == "cart_discount_amount_set":
        orders_df_sample[col] = orders_df_sample[col].apply(lambda x: float(x['shopMoney']['amount']) if x else 0)    
    else:
        orders_df_sample[col] = orders_df_sample[col].apply(lambda x: float(x))

# Columns with datetime values
date_cols = ['closed_at','created_at','cancelled_at']

for col in date_cols:
    col_utc = col + '_utc'

    if col == 'cancelled_at':
        orders_df_sample[col_utc] = orders_df_sample[col].apply(lambda x: pd.to_datetime(x) if x is not None else None)
    else:
        orders_df_sample[col_utc] = orders_df_sample[col].apply(lambda x: pd.to_datetime(x))

In [157]:
orders_df_sample.dtypes

order_id                                             object
order_name                                           object
email                                                object
created_at                                           object
cancellation                                         object
cancelled_at                                         object
cancel_reason                                        object
cart_discount_amount_set                            float64
channel_information                                  object
closed                                                 bool
closed_at                                            object
current_cart_discount_amount_set                    float64
current_shipping_price_set                          float64
current_subtotal_line_items_quantity                float64
current_subtotal_price_set                          float64
current_tax_lines                                    object
current_total_additional_fees_set       

In [202]:
# orders_df_sample.sample(n = 30)

In [203]:
# orders_df_sample['payment_gateway_names'].apply(lambda x: print(type(x)))

In [201]:
# orders_df_sample[['payment_gateway_names','discount_codes']].sort_values(by = 'payment_gateway_names', ascending= False)

In [140]:
# orders_df_sample.to_csv('table_sample_values_after_proc.csv')

Adding processed data to BQ:

In [200]:
# Need to create logic to differentiate dict objects since I can't directly pipe them into BigQuery. 

schema = [
    bigquery.SchemaField("order_id", "STRING"),
    bigquery.SchemaField("order_name", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("created_at", "STRING"),
    bigquery.SchemaField("cancellation", "STRING"),
    bigquery.SchemaField("cancelled_at", "STRING"),
    bigquery.SchemaField("cancel_reason", "STRING"),
    bigquery.SchemaField("cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("channel_information", "STRING"),
    bigquery.SchemaField("closed", "BOOLEAN"),
    bigquery.SchemaField("closed_at", "STRING"),
    bigquery.SchemaField("current_cart_discount_amount_set", "NUMERIC"),
    bigquery.SchemaField("current_shipping_price_set", "NUMERIC"),
    bigquery.SchemaField("current_subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("current_subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("current_tax_lines", "STRING"),
    bigquery.SchemaField("current_total_additional_fees_set", "STRING"),
    bigquery.SchemaField("current_total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("current_total_price_set", "NUMERIC"),
    bigquery.SchemaField("current_total_tax_set", "NUMERIC"),
    bigquery.SchemaField("discount_code", "STRING"),
    bigquery.SchemaField("discount_codes", "STRING"),
    bigquery.SchemaField("display_financial_status", "STRING"),
    bigquery.SchemaField("display_fulfillment_status", "STRING"),
    bigquery.SchemaField("fully_paid", "BOOLEAN"),
    bigquery.SchemaField("original_total_price_set", "NUMERIC"),
    bigquery.SchemaField("payment_gateway_names", "STRING"),
    bigquery.SchemaField("refunds", "NUMERIC"),
    bigquery.SchemaField("registered_source_url", "STRING"),
    bigquery.SchemaField("return_status", "STRING"),
    bigquery.SchemaField("source_name", "STRING"),
    bigquery.SchemaField("subtotal_line_items_quantity", "INTEGER"),
    bigquery.SchemaField("subtotal_price_set", "NUMERIC"),
    bigquery.SchemaField("total_discounts_set", "NUMERIC"),
    bigquery.SchemaField("total_price_set", "NUMERIC"),
    bigquery.SchemaField("transactions", "STRING"),

    bigquery.SchemaField("closed_at_utc", "DATETIME"),
    bigquery.SchemaField("created_at_utc", "DATETIME"),
    bigquery.SchemaField("cancelled_at_utc", "DATETIME"),
]
table_id_raw = "glowing-market-295819.shopify.test_table_orders_raw"

job_config = bigquery.LoadJobConfig(
schema = [
    bigquery.SchemaField("order_id", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("order_name", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("email", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("created_at", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancellation", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancelled_at", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cancel_reason", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("cart_discount_amount_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("channel_information", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closed", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("closed_at", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("current_cart_discount_amount_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("current_shipping_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("current_subtotal_line_items_quantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("current_subtotal_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("current_tax_lines", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("current_total_additional_fees_set", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("current_total_discounts_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("current_total_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("current_total_tax_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("discount_code", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("discount_codes", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("display_financial_status", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("display_fulfillment_status", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("fully_paid", bigquery.enums.SqlTypeNames.BOOLEAN),
    bigquery.SchemaField("original_total_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("payment_gateway_names", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("refunds", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("registered_source_url", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("return_status", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("source_name", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("subtotal_line_items_quantity", bigquery.enums.SqlTypeNames.INTEGER),
    bigquery.SchemaField("subtotal_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("total_discounts_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("total_price_set", bigquery.enums.SqlTypeNames.FLOAT64),
    bigquery.SchemaField("transactions", bigquery.enums.SqlTypeNames.STRING),
    bigquery.SchemaField("closed_at_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("created_at_utc", bigquery.enums.SqlTypeNames.DATETIME),
    bigquery.SchemaField("cancelled_at_utc", bigquery.enums.SqlTypeNames.DATETIME),
],
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(orders_df_sample,table_id_raw,job_config= job_config)
job.result()

table = client.get_table(table_id_raw)
print(
    "Loaded {} rows and {} columns to {}".format(
        table.num_rows, len(table.schema), table_id_raw
    )
)


/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 250 rows and 39 columns to glowing-market-295819.shopify.test_table_orders_raw


## Bulk Operations

In [20]:
bulk_query =  """
query { 
orders(query: "created_at:>2026-02-24") {
edges {
node {
id
name
email
createdAt
cancellation {
    staffNote
    } # closes cancellation
cancelledAt
cancelReason
cartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes cartDiscountAmountSet
channelInformation {
    displayName
    } # closes channelInformation
closed
closedAt
currentCartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentCartDiscountAmountSet
currentShippingPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentShippingPriceSet
currentSubtotalLineItemsQuantity
currentSubtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentSubtotalPriceSet
currentTaxLines {
    priceSet {
        shopMoney {
            amount
            } # closes shopMoney
        } # closes priceSet
    } # closes currentTaxLines 
currentTotalAdditionalFeesSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalAdditionalFeesSet
currentTotalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalDiscountsSet
currentTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalPriceSet
currentTotalTaxSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
discountCode
discountCodes
displayFinancialStatus
displayFulfillmentStatus
fullyPaid
originalTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
paymentGatewayNames
refunds {
    totalRefundedSet {
        shopMoney {
        amount
        } # closes shopMoney
    } # closes totalRefundedSet
    } # closes refunds
registeredSourceUrl
returnStatus
sourceName
subtotalLineItemsQuantity
subtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes subtotalPriceSet
totalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalDiscountsSet
totalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalPriceSet
transactions {
    device {
        id
        } # closes device
    } # closes transactions

} # Closing node
} # Closing edges
} # Closing orders
} # Closing query bracket
"""

Bulk Operation Query

In [24]:
mutation_q = """
        mutation {
  bulkOperationRunQuery(
   query: \"""
    query { 
orders(query: "created_at:>2026-02-24") {
edges {
node {
id
name
email
createdAt
cancellation {
    staffNote
    } # closes cancellation
cancelledAt
cancelReason
cartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes cartDiscountAmountSet
channelInformation {
    displayName
    } # closes channelInformation
closed
closedAt
currentCartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentCartDiscountAmountSet
currentShippingPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentShippingPriceSet
currentSubtotalLineItemsQuantity
currentSubtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentSubtotalPriceSet
currentTaxLines {
    priceSet {
        shopMoney {
            amount
            } # closes shopMoney
        } # closes priceSet
    } # closes currentTaxLines 
currentTotalAdditionalFeesSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalAdditionalFeesSet
currentTotalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalDiscountsSet
currentTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalPriceSet
currentTotalTaxSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
discountCode
discountCodes
displayFinancialStatus
displayFulfillmentStatus
fullyPaid
originalTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
paymentGatewayNames
refunds {
    totalRefundedSet {
        shopMoney {
        amount
        } # closes shopMoney
    } # closes totalRefundedSet
    } # closes refunds
registeredSourceUrl
returnStatus
sourceName
subtotalLineItemsQuantity
subtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes subtotalPriceSet
totalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalDiscountsSet
totalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalPriceSet
transactions {
    device {
        id
        } # closes device
    } # closes transactions

} # Closing node
} # Closing edges
} # Closing orders
} # Closing query bracket
    \"""
  ) {
    bulkOperation {
      id
      status
    }
    userErrors {
      field
      message
    }
  }
}
    """


In [ ]:
# mutation_q = """
#         mutation {
#   bulkOperationRunQuery(
#    query: \"""
#     {
#       products {
#         edges {
#           node {
#             id
#             title
#           }
#         }
#       }
#     }
#     \"""
#   ) {
#     bulkOperation {
#       id
#       status
#     }
#     userErrors {
#       field
#       message
#     }
#   }
# }
#     """


In [25]:
r = requests.post(
    f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
    headers={
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
    },
    json={"query": mutation_q}
)

bulk_query_response = r.json()

In [26]:
bulk_query_response

{'data': {'bulkOperationRunQuery': {'bulkOperation': {'id': 'gid://shopify/BulkOperation/7506460213616',
    'status': 'CREATED'},
   'userErrors': []}},
 'extensions': {'cost': {'requestedQueryCost': 10,
   'actualQueryCost': 10,
   'throttleStatus': {'maximumAvailable': 20000.0,
    'currentlyAvailable': 19990,
    'restoreRate': 1000.0}}}}

Getting Bulk Operation Status

In [27]:
q_bulk_status = """
query {
  currentBulkOperation {
    id
    status
    errorCode
    createdAt
    completedAt
    objectCount
    fileSize
    url
    partialDataUrl
  }
}
"""

In [30]:
r = requests.post(
    f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
    headers={
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
    },
    json={"query": q_bulk_status}
)

bulk_status_response = r.json()

In [31]:
bulk_query_response

{'data': {'bulkOperationRunQuery': {'bulkOperation': {'id': 'gid://shopify/BulkOperation/7506460213616',
    'status': 'CREATED'},
   'userErrors': []}},
 'extensions': {'cost': {'requestedQueryCost': 10,
   'actualQueryCost': 10,
   'throttleStatus': {'maximumAvailable': 20000.0,
    'currentlyAvailable': 19990,
    'restoreRate': 1000.0}}}}